In [70]:
import psycopg2
import os
import copy
import time
import pandas as pd
import numpy as np
import sys
sys.path.append("../")
from utils.load_cab_trace import convert_to_trace_df

In [50]:
host = "iconq-tpc-h.c39astlavjy2.us-east-1.rds.amazonaws.com"
port = "5432"
user = "postgres"
token = "postgres"
db_name = "tpc_sf10"
conn = psycopg2.connect(host=host, port=port, database=db_name, user=user, password=token)
conn.autocommit = True
cur = conn.cursor()

In [19]:
table_names = ["nation", "region", "part", "supplier", "partsupp", "customer", "orders", "lineitem"]
for table_name in table_names:
    cur.execute(f"SELECT COUNT(*) FROM {table_name};")
    res = cur.fetchall()
    print(table_name, res)

nation [(25,)]
region [(5,)]
part [(2000000,)]
supplier [(100000,)]
partsupp [(8000000,)]
customer [(1500000,)]
orders [(15000000,)]
lineitem [(59986052,)]


In [30]:
import json
with open("../../cab/benchmark-gen/query_streams/query_stream_0.json", "r") as f:
    p = json.load(f)
len(p['queries'])

2858

In [81]:
convert_to_trace_df("../../cab/benchmark-gen/query_streams/query_stream_0.json", "../workloads/cab/postgres_query_templates", 
                                                                 save_dir="../workloads/postgres/")

In [82]:
new_df = pd.read_csv("../workloads/postgres/tpc_sf1_query_trace.csv")
write_queries = new_df[new_df["query_template_idx"] == 23]

In [83]:
write_queries["g_offset_since_start_s"] = write_queries["g_offset_since_start_s"] * 1.6
write_queries

/var/folders/6m/hcldwlsj4k72ml4b0s9d5fqr0000gn/T/ipykernel_38438/2360462713.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  write_queries["g_offset_since_start_s"] = write_queries["g_offset_since_start_s"] * 1.6


,query_idx,run_time_s,g_offset_since_start_s,start_s,query_sql,query_template_idx
61,59,100.0,454.5280,40031.553,insert into orders (\n select o_orderkey + ...,23
69,67,100.0,524.3264,40075.177,insert into orders (\n select o_orderkey + ...,23
75,73,100.0,555.2080,40094.478,insert into orders (\n select o_orderkey + ...,23
79,77,100.0,579.6112,40109.730,insert into orders (\n select o_orderkey + ...,23
118,115,100.0,845.9536,40276.194,insert into orders (\n select o_orderkey + ...,23
...,...,...,...,...,...,...
2806,1686,100.0,63048.6224,79152.862,insert into orders (\n select o_orderkey + ...,23
2816,1693,100.0,63152.4672,79217.765,insert into orders (\n select o_orderkey + ...,23
2820,1697,100.0,63187.5168,79239.671,insert into orders (\n select o_orderkey + ...,23
2831,1702,100.0,63305.6768,79313.521,insert into orders (\n select o_orderkey + ...,23


In [99]:
1718 - 126

1592

In [100]:
write_queries["query_idx"] = np.arange(126) + 1592

/var/folders/6m/hcldwlsj4k72ml4b0s9d5fqr0000gn/T/ipykernel_38438/91480602.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  write_queries["query_idx"] = np.arange(126) + 1592


In [101]:
write_queries

,query_idx,run_time_s,g_offset_since_start_s,start_s,query_sql,query_template_idx
61,1592,100.0,454.5280,40031.553,insert into orders (\n select o_orderkey + ...,23
69,1593,100.0,524.3264,40075.177,insert into orders (\n select o_orderkey + ...,23
75,1594,100.0,555.2080,40094.478,insert into orders (\n select o_orderkey + ...,23
79,1595,100.0,579.6112,40109.730,insert into orders (\n select o_orderkey + ...,23
118,1596,100.0,845.9536,40276.194,insert into orders (\n select o_orderkey + ...,23
...,...,...,...,...,...,...
2806,1713,100.0,63048.6224,79152.862,insert into orders (\n select o_orderkey + ...,23
2816,1714,100.0,63152.4672,79217.765,insert into orders (\n select o_orderkey + ...,23
2820,1715,100.0,63187.5168,79239.671,insert into orders (\n select o_orderkey + ...,23
2831,1716,100.0,63305.6768,79313.521,insert into orders (\n select o_orderkey + ...,23


In [106]:
#res_df = pd.concat([df, write_queries])
res_df = res_df.sort_values("g_offset_since_start_s", ascending=True)

In [109]:
res_df.to_csv("../workloads/postgres/tpc_sf10_query_trace_write.csv", index=False)

In [98]:
query_file = "../workloads/postgres/tpc_sf10_all_unique_queries.sql"
with open(query_file, "r") as f:
    queries_text = f.read()
queries = queries_text.split(";\n\n")[:-1]
queries = [q.strip() + ";\n\n" for q in queries]
len(queries)

1718

In [84]:
write_queries["query_sql"].unique()[0]

'insert into orders (\n    select o_orderkey + 1000000000,\n           o_custkey,\n           o_orderstatus,\n           (select sum(L_QUANTITY * P_RETAILPRICE * (1+L_TAX) * (1-L_DISCOUNT)) from lineitem, part where l_orderkey = o_orderkey and P_PARTKEY = L_PARTKEY), o_orderdate, o_orderpriority, o_clerk, o_shippriority, o_comment\n    from orders\n    where 5994016 <= o_orderkey and o_orderkey < 6000032\n);\ndelete from orders where 5994016 + 1000000000 <= o_orderkey and o_orderkey < 6000032 + 1000000000;'

In [96]:
for q in write_queries["query_sql"].values:
    q += '\n\n'
    queries.append(q)
len(queries)

1718

In [97]:
with open("../workloads/postgres/tpc_sf10_all_unique_queries2.sql", "w") as f:
    for q in queries:
        f.write(q)

In [66]:
all_unique_queries = write_queries["query_sql"].unique()

In [68]:
len(all_unique_queries)

126

In [29]:
df = pd.read_csv("../workloads/postgres/tpc_sf10_query_trace.csv")
len(df)

2732

In [58]:
df

,query_idx,run_time_s,g_offset_since_start_s,start_s,query_sql,query_template_idx
0,0,1.0,0.000000,39747.473,select\n\tsum(l_extendedprice) / 7.0 as avg_ye...,17
1,1,1.0,11.495191,39747.525,select\n\tsum(l_extendedprice* (1 - l_discount...,19
2,2,1.0,19.612892,39747.943,"select\n\tl_shipmode,\n\tsum(case\n\t\twhen o_...",12
3,3,1.0,37.926380,39753.725,"select\n\tl_shipmode,\n\tsum(case\n\t\twhen o_...",12
4,4,1.0,49.291940,39754.063,"select\n\tc_name,\n\tc_custkey,\n\to_orderkey,...",18
...,...,...,...,...,...,...
2727,486,1.0,64850.862880,79461.176,with revenue_s as (\n\t\tselect\n\t\t\tl_suppk...,15
2728,1589,1.0,64863.031794,79465.728,"select\n\to_year,\n\tsum(case\n\t\twhen nation...",8
2729,1590,1.0,64876.150290,79467.815,"select\n\to_year,\n\tsum(case\n\t\twhen nation...",8
2730,1591,1.0,64898.761281,79477.049,"select\n\to_year,\n\tsum(case\n\t\twhen nation...",8


In [73]:
len(df["query_idx"].unique())

1592

In [57]:
print(new_df[new_df["query_template_idx"] == 23]["query_sql"].values[0])

insert into orders (
    select o_orderkey + 1000000000,
           o_custkey,
           o_orderstatus,
           (select sum(L_QUANTITY * P_RETAILPRICE * (1+L_TAX) * (1-L_DISCOUNT)) from lineitem, part where l_orderkey = o_orderkey and P_PARTKEY = L_PARTKEY), o_orderdate, o_orderpriority, o_clerk, o_shippriority, o_comment
    from orders
    where 5994016 <= o_orderkey and o_orderkey < 6000032
);

delete from orders where 5994016 + 1000000000 <= o_orderkey and o_orderkey < 6000032 + 1000000000;


In [52]:
sql = """

insert into orders (
    select o_orderkey + 1000000000,
           o_custkey,
           o_orderstatus,
           (select sum(L_QUANTITY * P_RETAILPRICE * (1+L_TAX) * (1-L_DISCOUNT)) from lineitem, part where l_orderkey = o_orderkey and P_PARTKEY = L_PARTKEY), o_orderdate, o_orderpriority, o_clerk, o_shippriority, o_comment
    from orders
    where 5994016 <= o_orderkey and o_orderkey < 6000032
);

delete from orders where 5994016 + 1000000000 <= o_orderkey and o_orderkey < 6000032 + 1000000000;

"""

cur.execute(sql)
#res = cur.fetchall()

In [46]:
res

[(1504,)]

In [47]:
cur.execute("select MAX(o_orderkey) from orders")
cur.fetchall()

[(60000000,)]

In [110]:
ENDPOINT="redshift-tpc-h.cmdzoy6ck5ua.us-east-1.redshift.amazonaws.com"
PORT="5439"
USER="awsuser"
REGION="us-east-1"
#DBNAME="dev"
DBNAME="tpc_sf81"
token="Giftedcoconut0!"
conn = psycopg2.connect(host=ENDPOINT, port=PORT, database=DBNAME, user=USER, password=token)
conn.autocommit = True
cur = conn.cursor()
cur.execute("SET enable_result_cache_for_session = OFF;")
conn.commit()

In [112]:
sql = """

insert into orders (
    select o_orderkey + 1000000000,
           o_custkey,
           o_orderstatus,
           (select sum(L_QUANTITY * P_RETAILPRICE * (1+L_TAX) * (1-L_DISCOUNT)) from lineitem, part where l_orderkey = o_orderkey and P_PARTKEY = L_PARTKEY), o_orderdate, o_orderpriority, o_clerk, o_shippriority, o_comment
    from orders
    where 5994016 <= o_orderkey and o_orderkey < 6000032
);

delete from orders where 5994016 + 1000000000 <= o_orderkey and o_orderkey < 6000032 + 1000000000;

"""

cur.execute(sql)

In [131]:
query_file = "../workloads/redshift/tpc_sf81_queries.sql"
with open(query_file, "r") as f:
    queries_text = f.read()
queries = queries_text.split(";")[:-1]
queries = [q.strip() + ";\n\n" for q in queries]
len(queries)

2190

In [116]:
df = pd.read_csv("../workloads/redshift/tpc_sf81_query_trace.csv")
print(len(df))
df

4054


,query_idx,run_time_s,g_offset_since_start_s,start_s,query_sql,query_template_idx
0,0,1.0,0.000000,1.722,select\n\tsum(l_extendedprice * l_discount) as...,6
1,1,1.0,23.459024,10.443,with revenue as (\n\t\tselect\n\t\t\tl_suppkey...,15
2,2,1.0,44.374793,14.511,"select\n\tl_shipmode,\n\tsum(case\n\t\twhen o_...",12
3,3,1.0,61.160072,16.292,"select\n\tc_custkey,\n\tc_name,\n\tsum(l_exten...",10
4,4,1.0,78.851289,19.537,"select\n\tc_name,\n\tc_custkey,\n\to_orderkey,...",18
...,...,...,...,...,...,...
4049,1163,1.0,126120.838266,86152.137,select\n\tsum(l_extendedprice * l_discount) as...,6
4050,1020,1.0,126175.040266,86206.339,select\n\t100.00 * sum(case\n\t\twhen p_type l...,14
4051,160,1.0,126206.212349,86218.386,"select\n\tps_partkey,\n\tsum(ps_supplycost * p...",11
4052,1353,1.0,126271.587349,86283.761,"select\n\tl_shipmode,\n\tsum(case\n\t\twhen o_...",12


In [118]:
convert_to_trace_df("../../cab/benchmark-gen/query_streams/query_stream_2.json", "../workloads/cab/redshift_query_templates", 
                                                                 save_dir="../workloads/snowset/")
new_df = pd.read_csv("../workloads/snowset/tpc_sf81_query_trace.csv")
print(len(new_df))
write_queries = new_df[new_df["query_template_idx"] == 23]


4241


,query_idx,run_time_s,g_offset_since_start_s,start_s,query_sql,query_template_idx
55,55,100.0,716.149,717.871,insert into orders (\n select o_orderkey + ...,23
92,92,100.0,1293.119,1294.841,insert into orders (\n select o_orderkey + ...,23
102,102,100.0,1445.035,1446.757,insert into orders (\n select o_orderkey + ...,23
211,200,100.0,3817.233,3818.955,insert into orders (\n select o_orderkey + ...,23
215,204,100.0,3858.815,3860.537,insert into orders (\n select o_orderkey + ...,23
...,...,...,...,...,...,...
4088,2318,100.0,81314.685,81316.407,insert into orders (\n select o_orderkey + ...,23
4152,2348,100.0,83156.041,83157.763,insert into orders (\n select o_orderkey + ...,23
4153,2349,100.0,83160.856,83162.578,insert into orders (\n select o_orderkey + ...,23
4159,2350,100.0,83360.857,83362.579,insert into orders (\n select o_orderkey + ...,23


In [135]:
write_queries["g_offset_since_start_s"] = write_queries["g_offset_since_start_s"] * 1.3
write_queries["query_idx"] = np.arange(187) + 2190
write_queries

/var/folders/6m/hcldwlsj4k72ml4b0s9d5fqr0000gn/T/ipykernel_38438/1555599784.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  write_queries["g_offset_since_start_s"] = write_queries["g_offset_since_start_s"] * 1.3
/var/folders/6m/hcldwlsj4k72ml4b0s9d5fqr0000gn/T/ipykernel_38438/1555599784.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  write_queries["query_idx"] = np.arange(187) + 2190


,query_idx,run_time_s,g_offset_since_start_s,start_s,query_sql,query_template_idx
55,2190,100.0,930.9937,717.871,insert into orders (\n select o_orderkey + ...,23
92,2191,100.0,1681.0547,1294.841,insert into orders (\n select o_orderkey + ...,23
102,2192,100.0,1878.5455,1446.757,insert into orders (\n select o_orderkey + ...,23
211,2193,100.0,4962.4029,3818.955,insert into orders (\n select o_orderkey + ...,23
215,2194,100.0,5016.4595,3860.537,insert into orders (\n select o_orderkey + ...,23
...,...,...,...,...,...,...
4088,2372,100.0,105709.0905,81316.407,insert into orders (\n select o_orderkey + ...,23
4152,2373,100.0,108102.8533,83157.763,insert into orders (\n select o_orderkey + ...,23
4153,2374,100.0,108109.1128,83162.578,insert into orders (\n select o_orderkey + ...,23
4159,2375,100.0,108369.1141,83362.579,insert into orders (\n select o_orderkey + ...,23


In [125]:
all_unique_queries = write_queries["query_sql"].unique()
len(all_unique_queries), len(all_unique_queries) + len(queries)

(187, 2377)

In [132]:
for q in write_queries["query_sql"].values:
    q += '\n\n'
    queries.append(q)
len(queries)

2377

In [133]:
with open("../workloads/redshift/tpc_sf81_queries_write.sql", "w") as f:
    for q in queries:
        f.write(q)

In [134]:
query_file = "../workloads/redshift/tpc_sf81_queries_write.sql"
with open(query_file, "r") as f:
    queries_text = f.read()
queries = queries_text.split(";\n\n")[:-1]
queries = [q.strip() + ";" for q in queries]
len(queries)

2377

In [140]:
print(queries[0])

select
	sum(l_extendedprice * l_discount) as revenue
from
	lineitem
where
	l_shipdate >= date '1995-01-01'
	and l_shipdate < dateadd(year, 1, cast('1995-01-01' as date))
	and l_discount between 3 - 0.01 and 3 + 0.01
	and l_quantity < 25;


In [136]:
res_df = pd.concat([df, write_queries])
res_df = res_df.sort_values("g_offset_since_start_s", ascending=True)

In [138]:
res_df.to_csv("../workloads/redshift/tpc_sf81_query_trace_write.csv", index=False)

In [ ]:
plans = load_json(parsed_queries_path, namespace=False)